# 02 — Kaplan–Meier and Multivariable Cox Analysis

Demonstrates the unadjusted KM comparison and an adjusted Cox model using synthetic data.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path("..")
df = pd.read_csv(ROOT / "data" / "synthetic_dlbcl_demo.csv")
df.head()


In [ ]:
from statsmodels.duration.survfunc import SurvfuncRight

plt.figure(figsize=(7,4.5))
for group, g in df.groupby("treatment"):
    sf = SurvfuncRight(g["survival_months"], g["death"])
    plt.step(sf.surv_times, sf.surv_prob, where="post", label=group)
plt.xlabel("Months")
plt.ylabel("Overall survival probability")
plt.title("Synthetic demonstration: Kaplan–Meier overall survival")
plt.legend()
plt.show()


In [ ]:
from statsmodels.duration.hazard_regression import PHReg

m = df.copy()
m["CIT"] = (m["treatment"]=="Chemoimmunotherapy").astype(int)
m["male"] = (m["sex"]=="Male").astype(int)
m["advanced_stage"] = (m["stage"]=="III-IV").astype(int)

X = m[["CIT","age","male","advanced_stage","comorbidity_ge1","b_symptoms"]].astype(float)
model = PHReg(m["survival_months"], X, status=m["death"])
result = model.fit()
print(result.summary())
